In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
import multiprocessing
# from multiprocess import Pool
import sys

from selectinf.group_lasso_query import (group_lasso,
                                         split_group_lasso)

from selectinf.Simulation.instance import (logistic_group_instance)

from selectinf.base import selected_targets, full_targets

In [2]:
def estimate_hess(X, Y, proportion=0.5):
    from sklearn.linear_model import LogisticRegression
    model_no_intercept = LogisticRegression(fit_intercept=False)
    model_no_intercept.fit(X, Y)
    beta_full = np.array(model_no_intercept.coef_).squeeze()
    #loglike = rr.glm.logistic(X, successes=Y, trials=np.ones(n))
    # For LASSO, this is the OLS solution on X_{E,U}
    #beta_full = restricted_estimator(loglike, np.array([True] * p))
    def pi_hess(x):
        return np.exp(x) / (1 + np.exp(x)) ** 2

    # Calculation the asymptotic covariance of the MLE
    W = np.diag(pi_hess(X @ beta_full))

    return X.T @ W @ X * (1 - proportion) / proportion

In [3]:
n=500
s=10
signal_fac=1
rho=0.5
level=0.90
p=200

In [4]:
inst = logistic_group_instance
signal = np.sqrt(signal_fac * 2 * np.log(p))

In [5]:
groups = np.arange(p//4).repeat(4)
X, Y, beta = inst(n=n,
                  p=p,
                  signal=signal,
                  sgroup=s,
                  groups=groups,
                  ndiscrete=20,
                  nlevels=5,
                  sdiscrete=s - 3,  # s-3, # How many discrete rvs are not null
                  equicorrelated=False,
                  rho=rho,
                  random_signs=True)[:3]

In [6]:
weight_frac = 1
sigma_ = np.std(Y)
weights = dict([(i, weight_frac * sigma_ * np.sqrt(2 * np.log(p))) for i in np.unique(groups)])

# To determine the randomization covariance
# Ignore this for now
hess = estimate_hess(X=X, Y=Y, proportion=0.5)

# Workflow of the method
![alt text](algo.png "Algorithm")
## Matrices for parametrizing the implied Gaussian
![alt text](impliedGauss.png "Matrices")

# Initialization of the object
1. `logistic()` is a static method within the `group_lasso()` class
2. Sets hyperparameters, e.g. randomization covariance, ridge penalty, specifies likelihood, etc.

In [7]:
conv = group_lasso.logistic(X=X,
                            successes=Y,
                            trials=np.ones(n),
                            groups=groups,
                            weights=weights,
                            useJacobian=True,
                            ridge_term=0.,
                            cov_rand=hess)

# Solving the penalized estimator
## This completes the estimation steps of the procedure:
![alt text](estimation.png "Algorithm")

In [8]:
signs, _ = conv.fit()

# Setting up the implied Gaussian in the selective likelihood
## This completes step 5
![alt text](impliedStep.png "Algorithm")
## and computes the following matrices
![alt text](impliedGauss.png "Algorithm")
## They depict the joint distribution of the unpenalized refitted estimator $\widehat{\beta}_{n,E}$
## and the optimization variable $\widehat{\gamma}_n$ that characterizes model selection

In [9]:
# TODO: Implement/define some objects as futures

In [10]:
conv.setup_inference(dispersion=1)

# Setting up the (unadjusted) distribution of the unpenalized estimator
## This computes quantities (e.g. covariance) of the unpenalized refitted estimator $\widehat{\beta}_{n,E}$

In [13]:
target_spec = selected_targets(conv.loglike,
                               conv.observed_soln,
                               dispersion=1)

aaa


In [14]:
target_spec.cov_target

array([[ 61.80675351,  24.88915696,  21.0845312 , ...,   8.00139535,
         -8.26367132,   0.64475819],
       [ 24.88915696,  82.20656295,  27.31115944, ...,   1.63727763,
         -7.82078302,  17.22222898],
       [ 21.0845312 ,  27.31115944,  47.23919092, ...,   0.8414747 ,
         -3.87397222,   9.13573998],
       ...,
       [  8.00139535,   1.63727763,   0.8414747 , ...,  64.1166614 ,
        -20.88117453,  -8.59040604],
       [ -8.26367132,  -7.82078302,  -3.87397222, ..., -20.88117453,
         59.91319883, -21.27948062],
       [  0.64475819,  17.22222898,   9.13573998, ...,  -8.59040604,
        -21.27948062,  56.22148372]])

# Solving estimating equations and perform inference
## This completes step 6-12
![alt text](estEqn.png "Algorithm")

In [15]:
result,_ = conv.inference(target_spec,
                        method='selective_MLE',
                        level=level)

# Results

In [16]:
pval = result['pvalue']
intervals = np.asarray(result[['lower_confidence',
                               'upper_confidence']])

In [17]:
pval

0      0.376660
1      0.606795
2      0.746963
3      0.654576
4      0.561350
         ...   
195    0.851421
196    0.440190
197    0.926332
198    0.807615
199    0.678582
Name: pvalue, Length: 200, dtype: float64